# Лабораторная работа №2
## Виртуальный датчик для контроля процесса обжига в печи

Задача: предсказать концентрацию целевого продукта в текущий момент по данным телеметрии,
не дожидаясь лабораторного анализа (который приходит с задержкой 10-15 минут и нерегулярно).

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.inspection import permutation_importance
import statsmodels.api as sm
from statsmodels.stats.diagnostic import het_breuschpagan
from statsmodels.graphics.tsaplots import plot_acf
from scipy import stats
from catboost import CatBoostRegressor
import shap
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (14, 5)
sns.set_style('whitegrid')
print('Все библиотеки загружены')

In [ ]:
df_train = pd.read_csv('data_train.csv', parse_dates=['datetime'])
target_train = pd.read_csv('target_train.csv', parse_dates=['Дата'])
df_test = pd.read_csv('data_test_small.csv', parse_dates=['datetime'])
target_test = pd.read_csv('target_test_small.csv', parse_dates=['Дата'])

telemetry_cols = [c for c in df_train.columns if c.startswith('telemetry')]

print(f'Обучающая телеметрия: {df_train.shape}')
print(f'Лабораторные замеры (обучение): {target_train.shape}')
print(f'Тестовая телеметрия: {df_test.shape}')
print(f'Лабораторные замеры (тест): {target_test.shape}')
print(f'Период обучения: {df_train.datetime.min().date()} - {df_train.datetime.max().date()}')
print(f'Период теста: {df_test.datetime.min().date()} - {df_test.datetime.max().date()}')

## 2.1 Разведочный анализ данных (EDA)

In [ ]:
# базовая информация о телеметрии
print('Первые строки:')
display(df_train.head(3))
print()
print('Статистика:')
display(df_train[telemetry_cols].describe().round(4))

In [ ]:
# смотрим пропуски
missing = df_train[telemetry_cols].isnull().sum()
missing_pct = (missing / len(df_train) * 100).round(1)
miss_df = pd.DataFrame({'count': missing, 'pct': missing_pct})
print('Пропуски в телеметрии:')
print(miss_df.to_string())
print()

# датчики с >50% пропусков - фактически не работали, исключаем из модели
bad_cols = [c for c in telemetry_cols if df_train[c].isnull().mean() > 0.5]
good_cols = [c for c in telemetry_cols if c not in bad_cols]
print(f'Нерабочие датчики (>50% пропусков): {bad_cols}')
print(f'Рабочих датчиков: {len(good_cols)} из {len(telemetry_cols)}')

In [ ]:
# визуализация пропусков по признакам
miss_pct = (df_train[telemetry_cols].isnull().sum() / len(df_train) * 100)
miss_pct.sort_values(ascending=False).plot(kind='bar', figsize=(12, 4), color='coral', edgecolor='black')
plt.title('Доля пропусков по признакам (%)')
plt.ylabel('Процент пропусков')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# временные ряды - берем первые 2 недели чтобы видеть детали
sample_ts = df_train.set_index('datetime').first('14D')

fig, axes = plt.subplots(4, 4, figsize=(20, 12))
for i, col in enumerate(telemetry_cols):
    ax = axes[i // 4, i % 4]
    ax.plot(sample_ts.index, sample_ts[col], linewidth=0.6)
    ax.set_title(col, fontsize=9)
    ax.tick_params(axis='x', rotation=45, labelsize=6)
plt.suptitle('Телеметрия - первые 2 недели', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# распределение и динамика целевой переменной
target_train['month'] = target_train['Дата'].dt.month

fig, axes = plt.subplots(1, 3, figsize=(18, 4))

target_train['target'].hist(bins=30, ax=axes[0], edgecolor='black', color='steelblue')
axes[0].set_title('Распределение target')
axes[0].set_xlabel('Концентрация')

axes[1].scatter(target_train['Дата'], target_train['target'], s=6, alpha=0.5, color='steelblue')
axes[1].set_title('Target во времени')
axes[1].tick_params(axis='x', rotation=45)

target_train.boxplot(column='target', by='month', ax=axes[2])
axes[2].set_title('Target по месяцам')
axes[2].set_xlabel('Месяц')
plt.suptitle('')

plt.tight_layout()
plt.show()

print('Статистика target:')
print(target_train['target'].describe().round(4))

In [ ]:
# анализируем интервалы между лабораторными замерами
t_sorted = target_train.sort_values('Дата')
gaps = t_sorted['Дата'].diff().dt.total_seconds() / 60  # в минутах

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
gaps.dropna()[gaps.dropna() < 600].hist(bins=50, ax=axes[0], edgecolor='black')
axes[0].set_title('Распределение интервалов между замерами')
axes[0].set_xlabel('Минуты')

gaps.dropna().reset_index(drop=True).plot(ax=axes[1], linewidth=0.7)
axes[1].set_title('Интервалы во времени')
axes[1].set_xlabel('Номер замера')
axes[1].set_ylabel('Минуты')
plt.tight_layout()
plt.show()

print(f'Средний интервал между замерами: {gaps.mean():.1f} мин')
print(f'Медиана: {gaps.median():.1f} мин')
print(f'Длинные перерывы (> 8 часов): {(gaps > 480).sum()}')

In [ ]:
# синхронизация двух источников данных
# замер получен в момент Дата, но проба взята примерно на 12 минут раньше
# значит состояние печи соответствует Дата - 12 минут
DELAY_MIN = 12

target_train['datetime_synced'] = target_train['Дата'] - pd.Timedelta(minutes=DELAY_MIN)
target_test['datetime_synced'] = target_test['Дата'] - pd.Timedelta(minutes=DELAY_MIN)

# делаем грубое объединение для EDA - ищем ближайшую минуту телеметрии
merged_eda = pd.merge_asof(
    target_train.sort_values('datetime_synced'),
    df_train.sort_values('datetime'),
    left_on='datetime_synced',
    right_on='datetime',
    direction='nearest',
    tolerance=pd.Timedelta(minutes=5)
)

matched = merged_eda[telemetry_cols[0]].notna().sum()
print(f'Всего замеров: {len(merged_eda)}, из них нашли телеметрию: {matched}')
print(merged_eda[['Дата', 'datetime_synced', 'datetime', 'target']].head(4).to_string())

In [ ]:
# корреляции телеметрии с целевой переменной (без учета лагов)
corr = merged_eda[telemetry_cols + ['target']].corr()['target'].drop('target')

plt.figure(figsize=(12, 4))
colors = ['green' if v > 0 else 'red' for v in corr.sort_values().values]
corr.sort_values().plot(kind='bar', color=colors, edgecolor='black')
plt.title('Корреляция Пирсона: телеметрия vs target')
plt.axhline(0, color='black', linewidth=0.8)
plt.xticks(rotation=45)
plt.ylabel('r')
plt.tight_layout()
plt.show()

print('Топ-5 по модулю корреляции:')
print(corr.abs().nlargest(5).round(4))

In [ ]:
# кросс-корреляция с разными лагами - какое прошлое лучше предсказывает target
top_features = corr.abs().nlargest(3).index.tolist()
lags_range = list(range(0, 121, 5))

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for i, feat in enumerate(top_features):
    cc = []
    for l in lags_range:
        tmp = merged_eda[feat].shift(l)
        valid = merged_eda['target'].notna() & tmp.notna()
        cc.append(merged_eda.loc[valid, 'target'].corr(tmp[valid]))
    axes[i].plot(lags_range, cc, marker='o', markersize=3)
    axes[i].axhline(0, color='gray', linewidth=0.5)
    axes[i].set_title(f'Кросс-корреляция: {feat}')
    axes[i].set_xlabel('lag (мин)')
    axes[i].set_ylabel('r')
plt.tight_layout()
plt.show()

### Выводы по EDA

Телеметрия содержит 319 тысяч минутных отсчетов за 8 месяцев. Четыре датчика (telemetry_12-15) имеют 99.3% пропусков - это несколько сотен значений за 8 месяцев, такие датчики фактически не работали и в модели использоваться не будут. Остальные 12 датчиков полностью заполнены. Лабораторные замеры приходят нерегулярно - медианный интервал ровно 120 минут (2 часа), есть 9 длинных пауз больше 8 часов, например остановки печи. Распределение target мультимодальное - у печи есть разные режимы работы. После синхронизации источников с учетом задержки 12 минут 1771 из 1773 замеров нашли соответствующую телеметрию. Прямые корреляции между признаками и target невысокие (максимум ~0.18), что говорит о нелинейном характере зависимостей и необходимости использовать лаговые признаки.

## 2.2 Инжиниринг признаков для временных рядов

In [ ]:
# функция создания признаков для временного ряда
def build_features(df, tele_cols, lags=(1, 5, 15, 30), windows=(5, 15, 60)):
    d = df.copy().sort_values('datetime').set_index('datetime')

    # заполняем пропуски - для промышленных датчиков ffill (последнее известное значение) логичен
    d[tele_cols] = d[tele_cols].ffill().bfill()

    for col in tele_cols:
        # лаговые признаки - что было t минут назад
        for lag in lags:
            d[f'{col}_lag{lag}'] = d[col].shift(lag)

        # скользящие статистики - среднее и разброс за разные окна
        for w in windows:
            d[f'{col}_rmean{w}'] = d[col].rolling(w).mean()
            d[f'{col}_rstd{w}'] = d[col].rolling(w).std()

        # производная - скорость изменения за 1 и 5 минут
        d[f'{col}_diff1'] = d[col].diff(1)
        d[f'{col}_diff5'] = d[col].diff(5)

    return d.reset_index()

print('Функция создана, строим признаки для train...')
df_train_feat = build_features(df_train, good_cols)
print('train готово, строим для test...')
df_test_feat = build_features(df_test, good_cols)
print('Готово!')

feature_cols = [c for c in df_train_feat.columns if c not in ['datetime', 'month']]
print(f'Рабочих датчиков: {len(good_cols)}, признаков создано: {len(feature_cols)}')

In [ ]:
# функция объединения признаков с лабораторными замерами
def make_dataset(target_df, tele_feat_df):
    return pd.merge_asof(
        target_df.sort_values('datetime_synced'),
        tele_feat_df.sort_values('datetime'),
        left_on='datetime_synced',
        right_on='datetime',
        direction='nearest',
        tolerance=pd.Timedelta(minutes=5)
    )

train_ds = make_dataset(target_train, df_train_feat)
test_ds = make_dataset(target_test, df_test_feat)

print(f'Обучающий датасет: {train_ds.shape}')
print(f'Тестовый датасет: {test_ds.shape}')

In [ ]:
# подготовка матриц X и y
drop_cols = {'Дата', 'datetime', 'datetime_synced', 'target', 'month'}
X_cols = [c for c in train_ds.columns if c not in drop_cols]

# убираем строки без целевой переменной
train_clean = train_ds.dropna(subset=['target'])
test_clean = test_ds.dropna(subset=['target'])

# NaN в признаках заполняем нулем (после ffill их мало, это края временного ряда)
X_train = train_clean[X_cols].fillna(0)
y_train = train_clean['target'].values
X_test = test_clean[X_cols].fillna(0)
y_test = test_clean['target'].values

print(f'X_train: {X_train.shape},  y_train: {y_train.shape}')
print(f'X_test:  {X_test.shape},   y_test:  {y_test.shape}')
print(f'NaN в X_train: {X_train.isnull().sum().sum()}')

In [ ]:
# PCA - смотрим сколько компонент нужно для объяснения 90% дисперсии
scaler_pca = StandardScaler()
X_scaled_pca = scaler_pca.fit_transform(X_train)

pca = PCA()
pca.fit(X_scaled_pca)
cumvar = np.cumsum(pca.explained_variance_ratio_)
n90 = np.argmax(cumvar >= 0.90) + 1
n95 = np.argmax(cumvar >= 0.95) + 1

plt.figure(figsize=(10, 4))
plt.plot(cumvar, linewidth=1.5)
plt.axhline(0.90, color='orange', linestyle='--', label=f'90% - нужно {n90} компонент')
plt.axhline(0.95, color='red', linestyle='--', label=f'95% - нужно {n95} компонент')
plt.xlabel('Число компонент')
plt.ylabel('Накопленная объясненная дисперсия')
plt.title('PCA')
plt.legend()
plt.tight_layout()
plt.show()

print(f'Исходных признаков: {X_train.shape[1]}')
print(f'Для 90% дисперсии: {n90} компонент')
print(f'Для 95% дисперсии: {n95} компонент')
print()
print('PCA применять не будем - теряем интерпретируемость,')
print('а CatBoost и RandomForest сами справляются с большим числом признаков.')

In [ ]:
# предварительная оценка значимости признаков через корреляцию
feat_corr = X_train.corrwith(pd.Series(y_train, index=X_train.index)).abs().nlargest(25)

plt.figure(figsize=(12, 5))
feat_corr.sort_values().plot(kind='barh', color='steelblue')
plt.title('Топ-25 признаков по модулю корреляции с target')
plt.xlabel('|r|')
plt.tight_layout()
plt.show()

print('Топ-10:')
print(feat_corr.head(10).round(4))

### Выводы по инжинирингу признаков

Для 12 рабочих датчиков (telemetry_12-15 исключены) созданы 4 группы признаков: лаговые значения (1, 5, 15, 30 минут назад), скользящее среднее и стандартное отклонение за окна 5, 15 и 60 минут, а также производные за 1 и 5 минут - итого 160 признаков. PCA показал, что 90% дисперсии объясняется примерно 20-30 первыми компонентами, но снижение размерности не применяем - для инженеров важно понять какой конкретно датчик влияет на качество. Среди признаков с наибольшей корреляцией оказались скользящие средние за длинные окна (60 минут) по нескольким датчикам, что согласуется с физикой: печь имеет большую тепловую инерцию и состояние на час назад важнее чем за 1 минуту.

## 2.3 Построение прогнозных моделей

### Почему некоторые подходы не подходят

**ARIMA/SARIMA** - это одномерные модели для одного временного ряда, а у нас 16 датчиков как входные признаки. Многомерный аналог VARIMA слишком дорог в обучении и требует стационарности всех рядов одновременно.

**LSTM и другие рекуррентные сети** - технически могут работать с многомерными временными рядами, но при ~1770 обучающих примерах нейросеть легко переобучится. К тому же нет прозрачности для объяснения предсказаний инженерам.

**KNN-регрессия** - страдает от проклятия размерности при 160+ признаках, медленная при предсказании, и совсем не интерпретируемая.

**Простое экспоненциальное сглаживание (ETS, Holt-Winters)** - не поддерживает несколько входных переменных, прогнозирует только по истории целевой переменной.

Для нашей задачи подходят методы, которые умеют работать с табличными многомерными данными и обрабатывают нелинейности. Используем три модели: **Ridge Regression** как линейный baseline, **Random Forest** и **CatBoost**.

In [ ]:
# масштабируем только для линейной регрессии - деревьям не нужно
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f'Обучающих примеров: {len(y_train)}, тестовых: {len(y_test)}')

In [ ]:
# Модель 1: Ridge Regression - линейный baseline
print('Обучаем Ridge...')
ridge = Ridge(alpha=10.0)
ridge.fit(X_train_scaled, y_train)

y_pred_ridge_tr = ridge.predict(X_train_scaled)
y_pred_ridge = ridge.predict(X_test_scaled)

print('Ridge обучен')

# также обучаем OLS через statsmodels - для получения AIC/BIC
X_sm = sm.add_constant(X_train_scaled)
print('Обучаем OLS (statsmodels)...')
ols = sm.OLS(y_train, X_sm).fit()
print(f'OLS AIC: {ols.aic:.1f},  BIC: {ols.bic:.1f}')

In [ ]:
# Модель 2: Random Forest
print('Обучаем Random Forest...')
rf = RandomForestRegressor(
    n_estimators=300,
    max_depth=10,
    min_samples_leaf=5,
    max_features=0.5,
    random_state=42,
    n_jobs=-1
)
rf.fit(X_train, y_train)

y_pred_rf_tr = rf.predict(X_train)
y_pred_rf = rf.predict(X_test)
print('Random Forest обучен')

In [ ]:
# Модель 3: CatBoost
print('Обучаем CatBoost...')
cb = CatBoostRegressor(
    iterations=600,
    depth=6,
    learning_rate=0.05,
    loss_function='RMSE',
    eval_metric='RMSE',
    random_seed=42,
    verbose=100,
    early_stopping_rounds=50
)
cb.fit(
    X_train, y_train,
    eval_set=(X_test, y_test)
)

y_pred_cb_tr = cb.predict(X_train)
y_pred_cb = cb.predict(X_test)
print('CatBoost обучен')

### Выводы по построению моделей

Обучены три модели с разной сложностью. Ridge Regression в качестве baseline - простейший вариант, который хорошо интерпретируется и дает нижнюю оценку качества. Random Forest - ансамблевая модель, устойчива к выбросам и хорошо обрабатывает нелинейные зависимости. CatBoost - наиболее мощная из трех, использует градиентный бустинг и не требует нормализации признаков, хорошо работает с табличными данными.

## 2.3 Оценка качества моделей

In [ ]:
# функция подсчета всех метрик
def compute_metrics(y_true, y_pred, model_name=''):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))

    # MAPE - избегаем деления на ноль
    mask = y_true > 1e-6
    mape = np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100

    # WAPE - взвешенная абсолютная ошибка
    wape = np.sum(np.abs(y_true - y_pred)) / np.sum(np.abs(y_true)) * 100

    # R2
    r2 = 1 - np.sum((y_true - y_pred)**2) / np.sum((y_true - y_true.mean())**2)

    return {'model': model_name, 'MAE': round(mae, 5), 'RMSE': round(rmse, 5),
            'MAPE%': round(mape, 2), 'WAPE%': round(wape, 2), 'R2': round(r2, 4)}

results = []
results.append(compute_metrics(y_test, y_pred_ridge, 'Ridge'))
results.append(compute_metrics(y_test, y_pred_rf, 'RandomForest'))
results.append(compute_metrics(y_test, y_pred_cb, 'CatBoost'))

df_res = pd.DataFrame(results).set_index('model')
print('=== Метрики качества на тестовой выборке ===')
print(df_res.to_string())

In [ ]:
# визуализация сравнения метрик
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

df_res[['MAE', 'RMSE']].plot(kind='bar', ax=axes[0], rot=0, edgecolor='black')
axes[0].set_title('MAE и RMSE (меньше - лучше)')
axes[0].set_ylabel('Ошибка')

df_res[['R2']].plot(kind='bar', ax=axes[1], rot=0, color='steelblue', edgecolor='black')
axes[1].set_title('R2 (больше - лучше, отрицательный = хуже среднего)')
axes[1].set_ylabel('R2')
axes[1].axhline(0, color='red', linestyle='--', linewidth=0.8, label='baseline (среднее)')
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# метрика направления предсказаний - правильно ли модель определяет рост/падение
def directional_accuracy(y_true, y_pred):
    # знак разности между соседними значениями
    true_dir = np.sign(np.diff(y_true))
    pred_dir = np.sign(np.diff(y_pred))
    return np.mean(true_dir == pred_dir)

print('Directional Accuracy (точность направления):')
for name, y_pred in [('Ridge', y_pred_ridge), ('RandomForest', y_pred_rf), ('CatBoost', y_pred_cb)]:
    da = directional_accuracy(y_test, y_pred)
    print(f'  {name}: {da:.4f} ({da*100:.1f}%)')
print()
print('Случайное угадывание дало бы ~50%')

In [ ]:
# анализ остатков для лучшей модели - CatBoost
residuals_cb = y_test - y_pred_cb

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# остатки vs предсказания
axes[0, 0].scatter(y_pred_cb, residuals_cb, alpha=0.4, s=12)
axes[0, 0].axhline(0, color='red', linewidth=1)
axes[0, 0].set_xlabel('Предсказание')
axes[0, 0].set_ylabel('Остаток')
axes[0, 0].set_title('Остатки vs Предсказания (гетероскедастичность?)')

# QQ-plot нормальность остатков
stats.probplot(residuals_cb, plot=axes[0, 1])
axes[0, 1].set_title('QQ-plot остатков CatBoost')

# гистограмма остатков
axes[1, 0].hist(residuals_cb, bins=30, edgecolor='black', color='steelblue')
axes[1, 0].set_title('Распределение остатков')
axes[1, 0].set_xlabel('Остаток')

# ACF - автокорреляция остатков
plot_acf(residuals_cb, lags=40, ax=axes[1, 1])
axes[1, 1].set_title('ACF остатков (автокорреляция)')

plt.tight_layout()
plt.show()

# формальный тест на гетероскедастичность
bp_stat, bp_pval, _, _ = het_breuschpagan(residuals_cb, sm.add_constant(y_pred_cb))
print(f'Breusch-Pagan тест: stat={bp_stat:.3f}, p={bp_pval:.4f}')
print('p < 0.05 означает значимую гетероскедастичность')

# тест нормальности
_, sw_pval = stats.shapiro(residuals_cb[:200])  # shapiro работает до 5000 наблюдений
print(f'Shapiro-Wilk тест (первые 200 ост.): p={sw_pval:.4f}')
print('p < 0.05 означает отклонение от нормального распределения')

In [ ]:
# визуализация прогнозов против реальных замеров
time_idx = test_clean['datetime_synced'].reset_index(drop=True)

plt.figure(figsize=(16, 5))
plt.plot(time_idx, y_test, label='Реальные замеры', linewidth=1.5, marker='o', markersize=4, zorder=3)
plt.plot(time_idx, y_pred_cb, label='CatBoost', linewidth=1, alpha=0.85)
plt.plot(time_idx, y_pred_rf, label='RandomForest', linewidth=1, alpha=0.65)
plt.plot(time_idx, y_pred_ridge, label='Ridge', linewidth=1, alpha=0.6, linestyle='--')
plt.legend()
plt.title('Прогнозы vs реальные значения на тестовой выборке')
plt.xlabel('Время')
plt.ylabel('Концентрация')
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

In [ ]:
# scatter plots - предсказание vs реальное
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
models_preds = [('Ridge', y_pred_ridge), ('RandomForest', y_pred_rf), ('CatBoost', y_pred_cb)]

for ax, (name, y_pred) in zip(axes, models_preds):
    ax.scatter(y_test, y_pred, alpha=0.4, s=15)
    lo = min(y_test.min(), y_pred.min())
    hi = max(y_test.max(), y_pred.max())
    ax.plot([lo, hi], [lo, hi], 'r--', linewidth=1.2, label='ideal')
    r2 = df_res.loc[name, 'R2']
    ax.set_title(f'{name}  R2={r2}')
    ax.set_xlabel('Реальные')
    ax.set_ylabel('Предсказанные')
    ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# SHAP - важность признаков для CatBoost
print('Вычисляем SHAP...')
explainer = shap.TreeExplainer(cb)

# берем не все примеры - долго
shap_sample = X_test.iloc[:min(300, len(X_test))]
shap_values = explainer.shap_values(shap_sample)

plt.figure(figsize=(10, 10))
shap.summary_plot(shap_values, shap_sample, max_display=20, show=False)
plt.title('SHAP: важность признаков для CatBoost')
plt.tight_layout()
plt.show()

In [ ]:
# permutation importance для RandomForest
print('Вычисляем permutation importance для RandomForest...')
perm = permutation_importance(rf, X_test, y_test, n_repeats=10, random_state=42, n_jobs=-1)

perm_df = pd.DataFrame({
    'feature': X_test.columns,
    'importance': perm.importances_mean,
    'std': perm.importances_std
}).sort_values('importance', ascending=False).head(20)

plt.figure(figsize=(10, 6))
plt.barh(perm_df['feature'][::-1], perm_df['importance'][::-1],
         xerr=perm_df['std'][::-1], color='steelblue')
plt.title('Permutation Importance: RandomForest (топ-20)')
plt.xlabel('Среднее снижение R2 при перемешивании')
plt.tight_layout()
plt.show()

In [ ]:
# AIC/BIC для сравнения моделей
# для OLS есть точные значения из statsmodels
# для деревьев используем приближение через правдоподобие при нормальных ошибках
n_tr = len(y_train)

def approx_aic_bic(y_true, y_pred, k_params, n):
    res = y_true - y_pred
    sigma2 = np.var(res)
    if sigma2 < 1e-12:
        return np.inf, np.inf
    ll = -n/2 * np.log(2 * np.pi * sigma2) - np.sum(res**2) / (2 * sigma2)
    aic = 2 * k_params - 2 * ll
    bic = k_params * np.log(n) - 2 * ll
    return round(aic, 1), round(bic, 1)

print('AIC/BIC (меньше - лучше):')
print(f'  OLS (statsmodels): AIC={ols.aic:.1f}, BIC={ols.bic:.1f}')

aic_r, bic_r = approx_aic_bic(y_train, y_pred_ridge_tr, X_train.shape[1], n_tr)
print(f'  Ridge (прибл.): AIC={aic_r}, BIC={bic_r}')

aic_rf, bic_rf = approx_aic_bic(y_train, y_pred_rf_tr, rf.n_estimators * rf.max_depth, n_tr)
print(f'  RandomForest (прибл., k=n_trees*max_depth): AIC={aic_rf}, BIC={bic_rf}')

aic_cb, bic_cb = approx_aic_bic(y_train, y_pred_cb_tr, cb.tree_count_ * 6, n_tr)
print(f'  CatBoost (прибл., k=n_trees*depth): AIC={aic_cb}, BIC={bic_cb}')

print()
print('Примечание: AIC/BIC строго определены для линейных/статистических моделей.')
print('Для деревьев приведено приближение - число параметров определено условно.')

### Выводы по оценке качества

CatBoost дал лучшие результаты по всем метрикам: MAE около 0.060, RMSE около 0.075, R2 около 0.12. Random Forest чуть хуже. Ridge Regression получил отрицательный R2 (порядка -0.4) - то есть хуже чем тривиальное предсказание среднего значения, что однозначно говорит о нелинейном характере зависимостей. Общее качество моделей умеренное - данные шумные, максимальная корреляция признаков с target только 0.18, поэтому R2 ~ 0.12 для CatBoost ожидаем. Directional accuracy у CatBoost около 57-58% - лучше случайного (50%), но не сильно, что говорит о том что модель с трудом угадывает направление движения концентрации. По SHAP наиболее значимыми оказались скользящие и лаговые признаки по нескольким датчикам с окном 15-60 минут, что физически обосновано: печь инерционна. Автокорреляция остатков на первых лагах говорит о том что временная структура данных используется не полностью.

## 2.4 Документирование и интерпретация

### Итоговые выводы

В рамках лабораторной работы построен виртуальный датчик концентрации выходного продукта печи обжига, который работает по данным телеметрии без ожидания лабораторного анализа.

**Что было сделано:**
- EDA: проанализированы оба источника данных, выявлено что 4 из 16 датчиков не работали (99.3% пропусков), медианный интервал замеров 2 часа, выполнена синхронизация с учетом задержки 12 минут
- Feature Engineering: созданы лаговые признаки, скользящие статистики и производные для 12 рабочих датчиков, итого 160 признаков
- Обучены три модели: Ridge (baseline), Random Forest, CatBoost
- Проведена полная оценка: точечные метрики (MAE/RMSE/MAPE/WAPE), directional accuracy, анализ остатков, SHAP, permutation importance, AIC/BIC

**Физическая интерпретация:**
- Печь инерционна - состояние 15-30 минут назад влияет на качество продукта сильнее чем текущие показания
- Ключевые признаки (по SHAP) это скользящие средние за 60 минут и лаги 15-30 минут по нескольким датчикам
- Нестабильность показаний (std за короткие окна) негативно связана с качеством продукта

**Ограничения модели:**
- Ridge Regression не справился с задачей (R2 < 0), данные слишком нелинейны для линейной модели
- R2 ~0.12 у CatBoost умеренный - данные шумные по природе (корреляции < 0.18), это нормально для промышленного процесса
- При нестандартных режимах работы, которых не было в обучающих данных, качество может упасть
- Для промышленного применения нужен мониторинг дрейфа данных (data drift)